In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
import math
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-12-16 20:54:15.294721


#### Functions

#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

# get the dob
int_dob = int(str_task[3:].split('_')[-1])
print(f'DOB: {int_dob}')


str_dirname_output = './output'

flt_prop_train = 0.6

flt_prop_valid = 0.2

flt_prop_test = 0.2

int_n_months_inform = 2

int_n_months_holdout = 4

Project: 20241112-simple-model-test
Task: 09_15_in_60
Subtask: 01_data_split
DOB: 60


#### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)

# get month of request
df['request_month'] = df['request_datetime'].apply(
    lambda x: f'{str(x)[:7]}-01',
)
df['request_month'] = pd.to_datetime(df['request_month']).dt.date
# sort
df.sort_values(by='request_month', ascending=True, inplace=True)

# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 8.88 s, sys: 3.83 s, total: 12.7 s
Wall time: 7.24 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,request_month
175,5716209,2021-07-30 13:53:28.0444771,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Washington,Franchise,Washington,True,...,0,2024-11-26 05:52:28.000933,1215,1,0,3,NaN,1.444396,0,2021-07-01
154,5702150,2021-07-30 10:22:57.8246513,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Indiana,Franchise,Indiana,True,...,0,2024-11-26 05:52:28.000933,1215,1,0,4,NaN,0.938360,1,2021-07-01
18,5702106,2021-07-27 12:40:50.0115498,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,0,2024-11-26 05:52:28.000933,1218,1,0,2,NaN,1.347324,0,2021-07-01
207,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Illinois,Franchise,Illinois,False,...,1,2024-11-26 05:52:28.000933,1215,1,1,7,0.113477,1.174497,0,2021-07-01
206,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,False,...,1,2024-11-26 05:52:28.000933,1215,1,1,7,0.105172,1.174497,0,2021-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93346,8359905,2024-11-07 04:33:53+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-06.gzip,0,0,Illinois,Franchise,Indiana,True,...,0,2024-11-26 05:52:28.000933,20,1,1,-1,NaN,1.221521,0,2024-11-01
93345,8359905,2024-11-07 04:33:53+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-06.gzip,1,1,Illinois,Franchise,Indiana,True,...,0,2024-11-26 05:52:28.000933,20,1,1,-1,0.139565,1.221521,0,2024-11-01
93643,8359885,2024-11-12 23:23:03+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-12.gzip,1,1,Indiana,Franchise,Indiana,True,...,0,2024-11-26 05:52:28.000933,14,1,0,4,0.083187,1.290518,0,2024-11-01
93286,8359691,2024-11-06 06:11:41+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-05.gzip,1,1,Missouri,Franchise,Missouri,False,...,0,2024-11-26 05:52:28.000933,21,1,0,2,0.093134,1.120000,0,2024-11-01


#### Remove too new from days on books (funded date - run date)

In [6]:
%%time

df = df[df['days_on_books'] >= int_dob].copy()
# show
df

CPU times: user 999 ms, sys: 784 ms, total: 1.78 s
Wall time: 1.78 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,request_month
175,5716209,2021-07-30 13:53:28.0444771,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Washington,Franchise,Washington,True,...,0,2024-11-26 05:52:28.000933,1215,1,0,3,NaN,1.444396,0,2021-07-01
154,5702150,2021-07-30 10:22:57.8246513,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Indiana,Franchise,Indiana,True,...,0,2024-11-26 05:52:28.000933,1215,1,0,4,NaN,0.938360,1,2021-07-01
18,5702106,2021-07-27 12:40:50.0115498,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,0,2024-11-26 05:52:28.000933,1218,1,0,2,NaN,1.347324,0,2021-07-01
207,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Illinois,Franchise,Illinois,False,...,1,2024-11-26 05:52:28.000933,1215,1,1,7,0.113477,1.174497,0,2021-07-01
206,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,False,...,1,2024-11-26 05:52:28.000933,1215,1,1,7,0.105172,1.174497,0,2021-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89597,8173711,2024-09-04 03:28:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-03.gzip,1,1,Maryland,Independent,Maryland,False,...,0,2024-11-26 05:52:28.000933,84,0,0,3,NaN,1.229124,0,2024-09-01
90088,8171521,2024-09-12 03:38:11+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-11.gzip,0,0,California,Franchise,California,True,...,0,2024-11-26 05:52:28.000933,76,1,1,0,NaN,1.082279,0,2024-09-01
89563,8173663,2024-09-03 22:20:34+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-03.gzip,0,0,Idaho,Franchise,Idaho,True,...,0,2024-11-26 05:52:28.000933,84,1,1,3,NaN,1.036325,1,2024-09-01
89749,8174035,2024-09-06 04:10:05+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-05.gzip,0,0,Arizona,Franchise,Arizona,False,...,0,2024-11-26 05:52:28.000933,82,1,1,2,NaN,1.402660,1,2024-09-01


#### Get the request months

In [7]:
list_request_month = list(df['request_month'].value_counts().index)
# sort
list_request_month = sorted(list_request_month)

# show
for a, request_month in enumerate(list_request_month):
    print(f'{a+1} - {request_month}')

1 - 2021-07-01
2 - 2021-08-01
3 - 2021-09-01
4 - 2021-10-01
5 - 2021-11-01
6 - 2021-12-01
7 - 2022-01-01
8 - 2022-02-01
9 - 2022-03-01
10 - 2022-04-01
11 - 2022-05-01
12 - 2022-06-01
13 - 2022-07-01
14 - 2022-08-01
15 - 2022-09-01
16 - 2022-10-01
17 - 2022-11-01
18 - 2022-12-01
19 - 2023-01-01
20 - 2023-02-01
21 - 2023-03-01
22 - 2023-04-01
23 - 2023-05-01
24 - 2023-06-01
25 - 2023-07-01
26 - 2023-08-01
27 - 2023-09-01
28 - 2023-10-01
29 - 2023-11-01
30 - 2023-12-01
31 - 2024-01-01
32 - 2024-02-01
33 - 2024-03-01
34 - 2024-04-01
35 - 2024-05-01
36 - 2024-06-01
37 - 2024-07-01
38 - 2024-08-01
39 - 2024-09-01
40 - 2024-11-01


#### Get the training months

In [8]:
int_n_request_months = len(list_request_month)
# subtract inform and holdout
int_n_training_months = int_n_request_months - int_n_months_holdout - int_n_months_inform

# get the number of months for training data set
int_n_months_training_data = math.ceil(int_n_training_months * flt_prop_train)
# get the months
list_request_months_training = list_request_month[:int_n_months_training_data]

# show
for a, request_month in enumerate(list_request_months_training):
    print(f'{a+1} - {request_month}')

1 - 2021-07-01
2 - 2021-08-01
3 - 2021-09-01
4 - 2021-10-01
5 - 2021-11-01
6 - 2021-12-01
7 - 2022-01-01
8 - 2022-02-01
9 - 2022-03-01
10 - 2022-04-01
11 - 2022-05-01
12 - 2022-06-01
13 - 2022-07-01
14 - 2022-08-01
15 - 2022-09-01
16 - 2022-10-01
17 - 2022-11-01
18 - 2022-12-01
19 - 2023-01-01
20 - 2023-02-01
21 - 2023-03-01


#### Get the validation months

In [9]:
# get the number of months for validation data set
int_n_months_validation_data = math.ceil(int_n_training_months * flt_prop_valid)
int_end_tmp = int_n_months_training_data + int_n_months_validation_data
# get the months
list_request_months_validation = list_request_month[int_n_months_training_data:int_end_tmp]

# show
for a, request_month in enumerate(list_request_months_validation):
    print(f'{a+1} - {request_month}')

1 - 2023-04-01
2 - 2023-05-01
3 - 2023-06-01
4 - 2023-07-01
5 - 2023-08-01
6 - 2023-09-01
7 - 2023-10-01


#### Get the test months

In [10]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data
int_n_months_test_data = int_n_training_months - int_start_tmp
# get the months
list_request_months_test = list_request_month[int_start_tmp:int_start_tmp+int_n_months_test_data]

# show
for a, request_month in enumerate(list_request_months_test):
    print(f'{a+1} - {request_month}')

1 - 2023-11-01
2 - 2023-12-01
3 - 2024-01-01
4 - 2024-02-01
5 - 2024-03-01
6 - 2024-04-01


#### Get the inform months

In [11]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data + int_n_months_test_data
# get the months
list_request_months_inform = list_request_month[int_start_tmp:int_start_tmp+int_n_months_inform]

# show
for a, request_month in enumerate(list_request_months_inform):
    print(f'{a+1} - {request_month}')

1 - 2024-05-01
2 - 2024-06-01


#### Get the holdout months

In [12]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data + int_n_months_test_data + int_n_months_inform
# get the months
list_request_months_holdout = list_request_month[int_start_tmp:]

# show
for a, request_month in enumerate(list_request_months_holdout):
    print(f'{a+1} - {request_month}')

1 - 2024-07-01
2 - 2024-08-01
3 - 2024-09-01
4 - 2024-11-01


#### Create tags

In [13]:
%%time

# train
df['train'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_training else 0,
)

CPU times: user 48.9 ms, sys: 374 μs, total: 49.3 ms
Wall time: 49 ms


In [14]:
%%time

# valid
df['valid'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_validation else 0,
)

CPU times: user 39.2 ms, sys: 0 ns, total: 39.2 ms
Wall time: 38.8 ms


In [15]:
%%time

# test
df['test'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_test else 0,
)

CPU times: user 38 ms, sys: 0 ns, total: 38 ms
Wall time: 37.5 ms


In [16]:
%%time

# test
df['inform'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_inform else 0,
)

CPU times: user 33.9 ms, sys: 779 μs, total: 34.7 ms
Wall time: 33.9 ms


In [17]:
%%time

# test
df['holdout'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_holdout else 0,
)

CPU times: user 37 ms, sys: 0 ns, total: 37 ms
Wall time: 36.2 ms


#### Write to s3

In [18]:
list_str_filename = [
    'train',
    'valid',
    'test',
    'inform',
    'holdout',
]
for str_filename in tqdm(list_str_filename):
    print(f'Filename: {str_filename}')
    # subset
    df_tmp = df[df[str_filename] == 1].copy()
    # get n rows
    int_nrows = df_tmp.shape[0]
    print(f'Rows: {int_nrows}')
    print()
    # make column name
    df_tmp['data_set'] = str_filename
    # write
    str_filename_tmp = f'df_{str_filename}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename_tmp}'
    df_tmp.to_parquet(str_uri, compression='gzip')

  0%|          | 0/5 [00:00<?, ?it/s]

Filename: train
Rows: 51178



 20%|██        | 1/5 [00:24<01:39, 24.87s/it]

Filename: valid
Rows: 16010



 40%|████      | 2/5 [00:32<00:44, 14.88s/it]

...

 60%|██████    | 3/5 [00:40<00:23, 11.65s/it]

Filename: inform
Rows: 3373



 80%|████████  | 4/5 [00:42<00:07,  7.86s/it]

Filename: holdout
Rows: 5014



100%|██████████| 5/5 [00:45<00:00,  9.09s/it]
